<a href="https://colab.research.google.com/github/BryanSqq/AI-learning-tasks/blob/main/AI_%E8%A8%98%E5%B8%B3%E5%8A%A9%E7%90%86_Web_App.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
pip install langgraph langchain-core langchain-google-genai

In [4]:
pip install --upgrade gradio

In [7]:
#函式注入
import os
import sqlite3
from pydantic import BaseModel, Field
from google.colab import userdata
from typing import Annotated, Literal
from typing_extensions import TypedDict

from langchain_core.messages import BaseMessage, ToolMessage
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import END, StateGraph, START
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

In [8]:
# 載入API key & 連結模型
os.environ["GOOGLE_API_KEY"] = userdata.get('GEMINI_API_KEY')
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [ ]:
import os
import sqlite3
import gradio as gr
from typing import Annotated, Literal
from typing_extensions import TypedDict
from pydantic import BaseModel, Field
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

# ==========================================
# 1. 初始化資料庫 (確保資料表存在)
# ==========================================
def init_db():
    conn = sqlite3.connect("expenses.db")
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS expenses (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            item TEXT,
            category TEXT,
            location TEXT,
            time TEXT,
            amount REAL
        )
    ''')
    conn.commit()
    conn.close()

init_db()

# ==========================================
# 2. 定義 Pydantic 規格與 LangChain 工具
# ==========================================
@tool
def record_expense(item: str, amount: float, category: str, location: str = "未知", time: str = "未知") -> str:
    """幫使用者記錄一筆消費紀錄到資料庫。"""
    conn = sqlite3.connect("expenses.db")
    cursor = conn.cursor()
    cursor.execute(
        "INSERT INTO expenses (item, category, location, time, amount) VALUES (?, ?, ?, ?, ?)",
        (item, category, location, time, amount)
    )
    conn.commit()
    conn.close()
    return f"【成功寫入資料庫】已記錄：在 {location} 購買 {item} 花費 {amount} 元，分類為【{category}】。"

class QueryExpenseInput(BaseModel):
    query_type: Literal["total", "category", "location", "time"] = Field(
        description="查詢的維度類型。如果想查全部總和請填 'total'；按分類查填 'category'；按地點查填 'location'；按時間查填 'time'。"
    )
    db_arg: str = Field(
        description="對應查詢的關鍵字。查 total 時請填空字串。"
    )

@tool(args_schema=QueryExpenseInput)
def query_expense(query_type: str, db_arg: str) -> str:
    """查詢花費的工具。當使用者要求知道花費總和或明細時使用。"""
    conn = sqlite3.connect("expenses.db")
    cursor = conn.cursor()

    try:
        if query_type == "total":
            cursor.execute("SELECT * FROM expenses")
        elif query_type in ["category", "location", "time"]:
            cursor.execute(f"SELECT * FROM expenses WHERE {query_type} = ?", [db_arg])

        rows = cursor.fetchall()

        if not rows:
            return "資料庫中沒有找到任何符合的資料。"

        result = "以下是查詢到的明細：\n"
        total = 0
        for row in rows:
            result += f"- [{row[4]}] 在 {row[3]} 買了 {row[1]} ({row[2]})，花費: {row[5]} 元\n"
            total += row[5]

        result += f"\n💰 此條件下的總花費為：{total} 元"
        return result
    finally:
        conn.close()

# ==========================================
# 3. 建立 LangGraph 自動化工作流
# ==========================================
# 確保你已經在 Colab 的 Secrets 中設定了 GEMINI_API_KEY
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
tools = [record_expense, query_expense]
llm_with_tools = llm.bind_tools(tools)

class State(TypedDict):
    messages: Annotated[list, add_messages]

workflow = StateGraph(State)

def chatbot(state: State):
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

workflow.add_node("agent", chatbot)
workflow.add_node("tools", ToolNode(tools=tools))
workflow.add_edge(START, "agent")
workflow.add_conditional_edges("agent", tools_condition)
workflow.add_edge("tools", "agent")

app = workflow.compile()

# ==========================================
# 4. 打造 Gradio Web 前端介面
# ==========================================
def chat_with_ai(message, history):
    """
    這個函式是 Gradio 與 LangGraph 的橋樑。
    它會接收使用者在網頁輸入的新訊息 (message)，以及網頁上的對話歷史 (history)。
    """
    print(f"Debug: history is {history}")
    # 步驟 A：將 Gradio 的歷史紀錄轉換成 LangGraph 認識的格式
    formatted_history = []
    for msg in history:
      print(f"Debug: msg is {msg}")
      if msg.get('content') and isinstance(msg['content'], list):
            # 取出第一個元素的 'text'，如果有多個內容塊可以考慮合併
            text_content = msg['content'][0].get('text', '')
            formatted_history.append({
                "role": msg['role'],
                "content": text_content
          })
      print(formatted_history)
    formatted_history.append({
        "role": "user",
        "content": message
    })
    # 步驟 C：啟動 LangGraph 引擎去思考與執行工具
    inputs = {"messages": formatted_history}
    response = app.invoke(inputs)

    # 步驟 D：抓取圖 (Graph) 跑完後，AI 最後一句白話文回覆，並顯示在網頁上
    final_ai_message = response["messages"][-1].content
    return final_ai_message

# 建立華麗的 Chatbot 介面
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.ChatInterface(
    fn=chat_with_ai,
    title="💰 你的專屬 AI 記帳管家",
    description="你好！我是你的 AI 記帳管家。你可以用口語對我說：「幫我記一下今天在小七買咖啡花了 55 元」，或是問我：「幫我算一下我目前總共花了多少錢？」",
    examples=[
        "我剛剛在麥當勞吃了大麥克套餐，花了 150 元",
        "昨天晚上去加油站加 95 汽油，花了 1200 元",
        "幫我算一下我目前總共花了多少錢？",
        "我最近在「交通」上總共花了多少？"
    ]
)

# 啟動網頁伺服器！
# share=True 會生成一個 public 網址 (如 https://xxxx.gradio.live) 讓你能在手機上測試
demo.launch(debug=True, share=True)

/tmp/ipykernel_3244/1059156478.py:145: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://a4a92554b05d080b6c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Debug: history is []
Debug: history is [{'role': 'user', 'metadata': None, 'content': [{'text': '我坐uber花了345元', 'type': 'text'}], 'options': None}, {'role': 'assistant', 'metadata': None, 'content': [{'text': '好的，已經幫您記錄一筆消費：搭乘 uber 花費 345 元，分類是交通。', 'type': 'text'}], 'options': None}]
Debug: msg is {'role': 'user', 'metadata': None, 'content': [{'text': '我坐uber花了345元', 'type': 'text'}], 'options': None}
[{'role': 'user', 'content': '我坐uber花了345元'}]
Debug: msg is {'role': 'assistant', 'metadata': None, 'content': [{'text': '好的，已經幫您記錄一筆消費：搭乘 uber 花費 345 元，分類是交通。', 'type': 'text'}], 'options': None}
[{'role': 'user', 'content': '我坐uber花了345元'}, {'role': 'assistant', 'content': '好的，已經幫您記錄一筆消費：搭乘 uber 花費 345 元，分類是交通。'}]
Debug: history is [{'role': 'user', 'metadata': None, 'content': [{'text': '我坐uber花了345元', 'type': 'text'}], 'options': None}, {'role': 'assistant', 'metadata': None, 'content': [{'text': '好的，已經幫您記錄一筆消費：搭乘 uber 花費 345 元，分類是交通。', 'type': 'text'}], 'options': None}, {'role': '

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py", line 3302, in _generate
    response: GenerateContentResponse = self.client.models.generate_content(
                                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/google/genai/models.py", line 6503, in generate_content
    return self._generate_content(
           ^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/google/genai/models.py", line 4954, in _generate_content
    response = self._api_client.request(
               ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/google/genai/_api_client.py", line 1618, in request
    response = self._request(http_request, http_options, stream=False)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/google/genai/_api_client.py", line 1409, in _req